# kg_third -- the third domain, source code

Self-contained. Imports stdlib + torch + `datasets` (HF, pre-installed on
Kaggle) only -- **nothing from the project repo, no ceq/, no data/, no
results/**. The arm is rebuilt inline from `ceq/arm_smprime.py`'s own
definitions and PROVEN against that source (Cell 2, the rebuild gate)
before any model is built or trained.

**The question, one kernel, one question.** On full TinyStories (repetition
0.7977x/0.7985x, below one) a hard-concrete gated arm beat a matched softmax
twin at 5/5 seeds by 0.2326-0.2622 nats (23x the 0.01-nat tie band). A
sibling lane is running the second domain (WikiText-103, encyclopedic
English) locally. **This notebook is the third domain: source code**
(codeparrot/codeparrot-clean-valid) -- long-range bracket/scope dependencies,
a byte distribution nothing like prose, structurally unlike both prior rows.

**Pre-registered branches:**
- **TRANSFERS** -- hard-concrete better by >0.01 nats at every seed.
- **DOES NOT TRANSFER** -- tie or softmax twin wins.
- **MIXED** -- effect inside the seed spread on this corpus.

No tuning. If the gated arm loses here, it loses -- that is reported exactly
like a win would be.

**Budget: 1 hour wall clock, Kaggle-authorised.** Cell 1 timestamps the
kernel's own start; the driver (Cell 7) calibrates seconds/step from a short
timed run and sizes the seed x step grid to the remaining budget, dropping
seeds (never truncating a run mid-seed) if the full 5-seed pre-registered
grid will not fit, and says exactly what it dropped.


In [ ]:
# --------------------------------------------------------------- provenance
import io, json, math, os, platform, random, sys, time

import torch

KERNEL_START = time.time()
BUDGET_S = 55 * 60   # 1h authorised, 5 min reserved for the summary/eval tail

print("python", sys.version.split()[0], "| torch", torch.__version__)
print("platform", platform.platform())

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# NEVER read the accelerator from a metadata request -- read what torch itself
# got handed, which is what the notebook actually trained on.
ACCEL = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"
print("device:", DEVICE, "| accelerator (from torch, not metadata):", ACCEL)
if DEVICE != "cuda":
    print("WARNING: no GPU visible -- every number below is a CPU number.")

# float32 ONLY. torch.polar rejects bfloat16 outright; float16 crashes at
# step 0 because arm_smprime's `_ctype` maps every non-float32 dtype to
# complex128, whose `.real` is float64, landing in an fp16-cast Linear.
DTYPE = torch.float32

WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)
LOG_PATH = os.path.join(WORK, "kg_third_results.jsonl")

def log_row(**kw):
    """Append one JSON line and flush -- INCREMENTAL, never rewritten. A cut
    session loses only the run in flight, never a row already landed."""
    row = dict(ts=time.strftime("%Y-%m-%dT%H:%M:%S"), device=DEVICE,
                accelerator=ACCEL, elapsed_s=round(time.time() - KERNEL_START, 1),
                **kw)
    with io.open(LOG_PATH, "a", encoding="utf-8") as fh:
        fh.write(json.dumps(row, default=str) + "\n")
    print("[log]", json.dumps(row, default=str))
    return row

def budget_left():
    return BUDGET_S - (time.time() - KERNEL_START)

print("results ->", LOG_PATH, "| budget_s =", BUDGET_S)
log_row(label="kernel_start", budget_s=BUDGET_S)

## Cell 2 -- the rebuild gate (C27 item 3), and PROOF that it rejects

`REFERENCE` below is transcribed VERBATIM from `ceq/arm_smprime.py` as read
on the authoring machine 2026-09-22 (lines 108-361: `magnitude`, `blend`,
`gate`, `path_product`, `hop`, `numerator`, `operator`, `readout`). `REBUILD`
is the SAME transcription, independently re-typed, and it is what Cell 3's
model actually uses. The gate compares them BITWISE (`torch.equal`) at
`(beta, qk) in {(0,0), (1,1), (1,0)}`, `g=1.0` fixed, on fixed random
`q, k, u, theta, v`.

**Then it breaks REBUILD on purpose** -- flips one sign inside
`path_product`'s cumulative product -- and asserts the SAME comparison now
FAILS. A gate never shown to reject is not a gate (this project has
catalogued sixteen checks that could not fail); this cell proves this one
can, before Cell 3 is allowed to run.


In [ ]:
NEG = float("-inf")
GDTYPE = torch.float64
GCDTYPE = torch.complex128

def _ctype(dtype):
    return torch.complex64 if dtype == torch.float32 else GCDTYPE

# =============================================================== REFERENCE
# Transcribed verbatim from ceq/arm_smprime.py (magnitude, blend, gate,
# path_product, hop, numerator, operator, readout). qk-on softmax-corner
# defaults (beta=qk=g=1.0) match the module's own signatures.

def ref_magnitude(u):
    return torch.clamp(u, 0.0, 1.0)

def ref_blend(u, theta, g):
    m = ref_magnitude(torch.lerp(torch.ones_like(u), u, g))
    return m, theta * g

def ref_gate(m, theta):
    return m.to(_ctype(m.dtype)) * torch.polar(torch.ones_like(theta), theta)

def ref_path_product(a):
    s = a.shape[-1]
    idx = torch.arange(s, device=a.device)
    le = idx.unsqueeze(-1) >= idx.unsqueeze(-2)
    rows = torch.where(le, a.unsqueeze(-2), torch.ones_like(a).unsqueeze(-2))
    q = torch.flip(torch.cumprod(torch.flip(rows, [-1]), -1), [-1])
    g = torch.cat([q[..., 1:], torch.ones_like(q[..., :1])], -1)
    return g.masked_fill(~le, 0)

def ref_hop(u, theta, *, g=1.0, route="product"):
    m, th = ref_blend(u, theta, g)
    return ref_path_product(ref_gate(m, th)), ref_path_product(m)

def ref_numerator(q, k, u, theta, *, qk=1.0, g=1.0):
    n = q.shape[-2]
    w = qk * ((q @ k.transpose(-2, -1)) / math.sqrt(q.shape[-1]))
    up = torch.ones(n, n, dtype=torch.bool, device=q.device).triu(1)
    e = torch.exp(w.masked_fill(up, NEG))
    gh, rh = ref_hop(u, theta, g=g)
    live = rh > 0
    e = torch.where(live, e, torch.zeros_like(e))
    zero = torch.zeros_like(gh.imag)
    im = torch.where(gh.imag == 0, zero, gh.imag * e)
    return torch.complex(gh.real * e, im), rh * e

def ref_operator(q, k, u, theta, *, beta=1.0, qk=1.0, g=1.0):
    num, mod = ref_numerator(q, k, u, theta, qk=qk, g=g)
    zb = mod.sum(-1, keepdim=True) ** beta
    return torch.complex(num.real / zb, num.imag / zb)

def ref_readout(q, k, v, u, theta, *, beta=1.0, qk=1.0, g=1.0):
    a = ref_operator(q, k, u, theta, beta=beta, qk=qk, g=g)
    return a @ v.to(a.dtype)

# =============================================================== REBUILD
# The independently re-typed copy: THIS is what Cell 3's GatedBlock calls.
# Bitwise identical to REFERENCE if both are honest transcriptions of the
# same source -- which is exactly what the gate below checks.

def magnitude(u):
    return torch.clamp(u, 0.0, 1.0)

def blend(u, theta, g):
    m = magnitude(torch.lerp(torch.ones_like(u), u, g))
    return m, theta * g

def gate_fn(m, theta):
    return m.to(_ctype(m.dtype)) * torch.polar(torch.ones_like(theta), theta)

def path_product(a, _sign=1.0):
    """`_sign` is a rebuild-gate knob ONLY -- 1.0 is the honest rebuild;
    -1.0 flips one sign inside the cumulative product (the deliberate
    break), and is never passed by anything except the gate's own
    break-and-reject check below."""
    s = a.shape[-1]
    idx = torch.arange(s, device=a.device)
    le = idx.unsqueeze(-1) >= idx.unsqueeze(-2)
    rows = torch.where(le, a.unsqueeze(-2), torch.ones_like(a).unsqueeze(-2))
    q = torch.flip(torch.cumprod(torch.flip(rows, [-1]), -1), [-1])
    q = q * _sign          # <-- the single flipped sign, ==1.0 in production
    g = torch.cat([q[..., 1:], torch.ones_like(q[..., :1])], -1)
    return g.masked_fill(~le, 0)

def hop(u, theta, *, g=1.0, route="product", _break=False):
    m, th = blend(u, theta, g)
    sign = -1.0 if _break else 1.0
    return path_product(gate_fn(m, th), sign), path_product(m, sign)

def numerator(q, k, u, theta, *, qk=1.0, g=1.0, _break=False):
    n = q.shape[-2]
    w = qk * ((q @ k.transpose(-2, -1)) / math.sqrt(q.shape[-1]))
    up = torch.ones(n, n, dtype=torch.bool, device=q.device).triu(1)
    e = torch.exp(w.masked_fill(up, NEG))
    gh, rh = hop(u, theta, g=g, _break=_break)
    live = rh > 0
    e = torch.where(live, e, torch.zeros_like(e))
    zero = torch.zeros_like(gh.imag)
    im = torch.where(gh.imag == 0, zero, gh.imag * e)
    return torch.complex(gh.real * e, im), rh * e

def operator_fn(q, k, u, theta, *, beta=1.0, qk=1.0, g=1.0, _break=False):
    num, mod = numerator(q, k, u, theta, qk=qk, g=g, _break=_break)
    zb = mod.sum(-1, keepdim=True) ** beta
    return torch.complex(num.real / zb, num.imag / zb)

def readout_fn(q, k, v, u, theta, *, beta=1.0, qk=1.0, g=1.0, _break=False):
    a = operator_fn(q, k, u, theta, beta=beta, qk=qk, g=g, _break=_break)
    return a @ v.to(a.dtype)

# =============================================================== THE GATE

def _draw_gate_inputs(seed=15, b=2, h=2, s=16, d=8):
    g = torch.Generator().manual_seed(seed)
    shape = (b, h, s, d)
    q = torch.randn(shape, generator=g, dtype=GDTYPE)
    k = torch.randn(shape, generator=g, dtype=GDTYPE)
    v = torch.randn(shape, generator=g, dtype=GDTYPE)
    u = torch.rand(b, h, s, generator=g, dtype=GDTYPE)          # in [0,1)
    theta = (torch.rand(b, h, s, generator=g, dtype=GDTYPE) - 0.5) * 2 * math.pi
    return q, k, v, u, theta

def run_gate(_break=False):
    q, k, v, u, theta = _draw_gate_inputs()
    corners = [(0.0, 0.0), (1.0, 1.0), (1.0, 0.0)]
    all_ok = True
    detail = []
    for beta, qk in corners:
        r_m = ref_magnitude(u)
        b_m = magnitude(u)
        r_gh, r_rh = ref_hop(u, theta, g=1.0)
        b_gh, b_rh = hop(u, theta, g=1.0, _break=_break)
        r_op = ref_operator(q, k, u, theta, beta=beta, qk=qk, g=1.0)
        b_op = operator_fn(q, k, u, theta, beta=beta, qk=qk, g=1.0, _break=_break)
        r_ro = ref_readout(q, k, v, u, theta, beta=beta, qk=qk, g=1.0)
        b_ro = readout_fn(q, k, v, u, theta, beta=beta, qk=qk, g=1.0, _break=_break)
        ok = (torch.equal(r_m, b_m) and torch.equal(r_gh, b_gh) and
              torch.equal(r_rh, b_rh) and torch.equal(r_op, b_op) and
              torch.equal(r_ro, b_ro))
        detail.append((beta, qk, ok))
        all_ok = all_ok and ok
    return all_ok, detail

print("=== GATE, honest rebuild (must PASS) ===")
pass_ok, pass_detail = run_gate(_break=False)
for beta, qk, ok in pass_detail:
    print(f"  (beta={beta}, qk={qk}): {'PASS' if ok else 'FAIL'}")
assert pass_ok, "REBUILD does not match REFERENCE bitwise on the honest path -- STOP, the transcription is wrong"
log_row(label="gate_honest", pass_all=pass_ok, corners=str(pass_detail))

print("\\n=== GATE, DELIBERATELY BROKEN rebuild (must REJECT) ===")
break_ok, break_detail = run_gate(_break=True)
for beta, qk, ok in break_detail:
    print(f"  (beta={beta}, qk={qk}): {'PASS (WRONG -- gate is inert)' if ok else 'REJECTED (correct)'}")
assert not break_ok, "the broken rebuild passed the gate -- the gate cannot fail and is worthless"
log_row(label="gate_break_and_reject", any_false_pass=break_ok, corners=str(break_detail))

print("\\nGATE VERIFIED: passes the honest rebuild bitwise, REJECTS a one-sign break. "
      "Safe to build the model below.")

## Cell 3 -- the two arms, matching `ceq/hf/modeling_ceq.py` exactly

`CEQAttention`/`CEQBlock`/`CEQModel`/`CEQForCausalLM` rebuilt inline:
pre-LN, `qkv`/`o_proj` (bias=False), MLP `d -> 4d -> d` (GELU, bias=True),
tied `lm_head`. At `operator="smprime"` the block additionally carries
`m_head`/`theta_head` (`nn.Linear(d,1)`) and three scalar switches
`beta, qk, g` (`nn.Parameter`, started at the arm's own softmax corner
`(1.0, 1.0, 1.0)`) -- `2*(d+1)+3` extra parameters per block, exactly the
783 = `3*(2*129+3)` the pre-registration names. `magnitude_hardconcrete`
is `tests/chase/gate/r1_gate.py`'s Louizos/Welling/Kingma 2018 form
(`zeta=1.1, gamma=-0.1`); `build_repaired`'s gate init
(`m_head.bias=1-1e-3, theta_head.bias=1e-3`) is applied after construction,
matching `ceq/arm_smprime.py::GATE_INIT_OFF` and the sibling lane's own row.


In [ ]:
INITIALIZER_RANGE = 0.02
GATE_INIT_OFF = 1e-3
ZETA, GAMMA = 1.1, -0.1

def magnitude_hardconcrete(u):
    s = torch.sigmoid(u)
    sbar = s * (ZETA - GAMMA) + GAMMA
    return torch.clamp(sbar, 0.0, 1.0)

def _init_weights(module):
    if isinstance(module, torch.nn.Linear):
        module.weight.data.normal_(mean=0.0, std=INITIALIZER_RANGE)
        if module.bias is not None:
            module.bias.data.zero_()
    elif isinstance(module, torch.nn.Embedding):
        module.weight.data.normal_(mean=0.0, std=INITIALIZER_RANGE)
    elif isinstance(module, torch.nn.LayerNorm):
        module.weight.data.fill_(1.0)
        module.bias.data.zero_()

class SoftmaxAttention(torch.nn.Module):
    """arm (a) -- genuine causal softmax attention, qkv/o_proj skeleton
    identical to CEQAttention's non-smprime branch."""
    def __init__(self, d, n_heads):
        super().__init__()
        assert d % n_heads == 0 and (d // n_heads) % 8 == 0
        self.n_heads, self.d_head = n_heads, d // n_heads
        self.qkv = torch.nn.Linear(d, 3 * d, bias=False)
        self.o_proj = torch.nn.Linear(d, d, bias=False)

    def forward(self, x):
        b, s, d = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        shape = lambda t: t.view(b, s, self.n_heads, self.d_head).transpose(1, 2)
        q, k, v = shape(q), shape(k), shape(v)
        o = torch.nn.functional.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o_proj(o.transpose(1, 2).reshape(b, s, d))

class SMPrimeAttention(torch.nn.Module):
    """arm (f) -- ceq/arm_smprime.py's readout, via Cell 2's PROVEN REBUILD."""
    def __init__(self, d, n_heads, magnitude_fn):
        super().__init__()
        assert d % n_heads == 0 and (d // n_heads) % 8 == 0
        self.n_heads, self.d_head = n_heads, d // n_heads
        self.magnitude_fn = magnitude_fn
        self.qkv = torch.nn.Linear(d, 3 * d, bias=False)
        self.o_proj = torch.nn.Linear(d, d, bias=False)
        self.m_head = torch.nn.Linear(d, 1)
        self.theta_head = torch.nn.Linear(d, 1)
        self.beta = torch.nn.Parameter(torch.tensor(1.0))
        self.qk = torch.nn.Parameter(torch.tensor(1.0))
        self.g = torch.nn.Parameter(torch.tensor(1.0))

    def forward(self, x):
        b, s, d = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        shape = lambda t: t.view(b, s, self.n_heads, self.d_head).transpose(1, 2)
        q, k, v = shape(q), shape(k), shape(v)
        u = self.m_head(x).squeeze(-1).unsqueeze(-2)         # [b, 1, s]
        th = self.theta_head(x).squeeze(-1).unsqueeze(-2)
        global magnitude
        _saved = magnitude
        magnitude = self.magnitude_fn                        # blend() reads the module global
        try:
            o = readout_fn(q, k, v, u, th, beta=self.beta, qk=self.qk, g=self.g).real
        finally:
            magnitude = _saved
        return self.o_proj(o.transpose(1, 2).reshape(b, s, d))

class CEQBlockInline(torch.nn.Module):
    def __init__(self, d, n_heads, attn):
        super().__init__()
        self.input_layernorm = torch.nn.LayerNorm(d, eps=1e-5)
        self.post_attention_layernorm = torch.nn.LayerNorm(d, eps=1e-5)
        self.self_attn = attn
        self.mlp = torch.nn.Sequential(torch.nn.Linear(d, 4 * d), torch.nn.GELU(),
                                        torch.nn.Linear(4 * d, d))

    def forward(self, x):
        x = x + self.self_attn(self.input_layernorm(x))
        return x + self.mlp(self.post_attention_layernorm(x))

class CEQForCausalLMInline(torch.nn.Module):
    def __init__(self, d, n_layers, n_heads, seq, vocab, arm):
        super().__init__()
        assert arm in ("softmax", "smprime")
        self.arm = arm
        self.embed_tokens = torch.nn.Embedding(vocab, d)
        self.embed_positions = torch.nn.Embedding(seq, d)
        block_factory = ((lambda: CEQBlockInline(d, n_heads, SoftmaxAttention(d, n_heads)))
                          if arm == "softmax" else
                          (lambda: CEQBlockInline(d, n_heads, SMPrimeAttention(d, n_heads, magnitude_hardconcrete))))
        self.layers = torch.nn.ModuleList(block_factory() for _ in range(n_layers))
        self.norm = torch.nn.LayerNorm(d, eps=1e-5)
        self.lm_head = torch.nn.Linear(d, vocab, bias=False)
        self.apply(_init_weights)
        self.lm_head.weight = self.embed_tokens.weight     # tied, +7.9% params avoided
        if arm == "smprime":
            off = GATE_INIT_OFF
            with torch.no_grad():
                for layer in self.layers:
                    layer.self_attn.m_head.bias.fill_(1.0 - off)
                    layer.self_attn.theta_head.bias.fill_(off)

    def n_params(self):
        return sum(p.numel() for p in self.parameters())

    def forward(self, input_ids, labels=None):
        b, s = input_ids.shape
        x = self.embed_tokens(input_ids) + self.embed_positions(torch.arange(s, device=input_ids.device))[None]
        for layer in self.layers:
            x = layer(x)
        x = self.norm(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = torch.nn.functional.cross_entropy(
                logits[:, :-1].reshape(-1, logits.shape[-1]).float(),
                labels[:, 1:].reshape(-1))
        return logits, loss

def build_arm(arm, *, hidden_size, n_layers, n_heads, seq, vocab_size=256):
    return CEQForCausalLMInline(hidden_size, n_layers, n_heads, seq, vocab_size, arm)

HIDDEN, LAYERS, HEADS, SEQ, BATCH, VOCAB = 128, 3, 8, 512, 8, 256
LR = 3e-4
SPLIT_SEED = 0
EVAL_BATCHES = 8

torch.manual_seed(0)
_m_a = build_arm("softmax", hidden_size=HIDDEN, n_layers=LAYERS, n_heads=HEADS, seq=SEQ, vocab_size=VOCAB)
N_PARAMS_A = _m_a.n_params()
del _m_a
torch.manual_seed(0)
_m_f = build_arm("smprime", hidden_size=HIDDEN, n_layers=LAYERS, n_heads=HEADS, seq=SEQ, vocab_size=VOCAB)
N_PARAMS_F = _m_f.n_params()
del _m_f

print(f"n_params softmax(a)={N_PARAMS_A:,}  hard-concrete(f)={N_PARAMS_F:,}  "
      f"diff={N_PARAMS_F - N_PARAMS_A} ({100.0*(N_PARAMS_F-N_PARAMS_A)/N_PARAMS_A:.4f}%)  "
      f"expected diff = 3*(2*129+3) = {3*(2*129+3)}")
log_row(label="param_check", n_params_a=N_PARAMS_A, n_params_f=N_PARAMS_F,
        diff=N_PARAMS_F - N_PARAMS_A, expected_diff=3*(2*129+3))

## Cell 4 -- the third corpus: source code, pinned and printed

`codeparrot/codeparrot-clean-valid` (accessible from Kaggle notebooks with
internet on, no credentials -- verified in the PREPARE step; 61,373 rows,
142MB, mixed per-file licences: MIT/Apache-2.0/GPL-2.0/GPL-3.0/BSD-2/BSD-3/
ISC/AGPL-3.0, no single unified licence declared). Long-range
bracket/scope dependencies and a byte distribution unlike prose or
encyclopedic English -- structurally the third domain. Streamed, capped at
`MAX_BYTES` (same order as the TinyStories/WikiText rows), split BY
DOCUMENT (one row = one file = one document), `val_frac=0.1`,
`split_seed=0` fixed across seeds -- byte-identical protocol to
`scratchpad/q2_second.py`.

If codeparrot is unreachable at runtime, this cell falls back to
`vikp/python_code_instructions_18k` (also public, also code, no
credentials) and says so explicitly rather than silently substituting.


In [ ]:
from datasets import load_dataset
from collections import Counter

MAX_BYTES = 20 * 1024 * 1024   # ~TinyStories row's own order of magnitude

def _row_text(row):
    for key in ("content", "text", "code", "whole_func_string"):
        v = row.get(key)
        if v:
            return v
    return None

def load_code_corpus(max_bytes=MAX_BYTES, max_rows=200000):
    tried = []
    for name, split, licence_note in [
        ("codeparrot/codeparrot-clean-valid", "train",
         "Mixed per-file licences (MIT/Apache-2.0/GPL-2.0/GPL-3.0/BSD-2-Clause/"
         "BSD-3-Clause/ISC/AGPL-3.0); no single unified dataset licence declared "
         "(source: huggingface.co/datasets/codeparrot/codeparrot-clean-valid)."),
        ("vikp/python_code_instructions_18k", "train",
         "FALLBACK corpus (codeparrot was unreachable) -- check its own card "
         "for licence terms before any downstream use beyond this measurement."),
    ]:
        try:
            ds = load_dataset(name, split=split, streaming=True)
            docs, total = [], 0
            license_counts = Counter()
            n_rows = 0
            for row in ds:
                n_rows += 1
                if n_rows > max_rows:
                    break
                t = _row_text(row)
                if not t:
                    continue
                docs.append(t)
                total += len(t.encode("utf-8", errors="ignore"))
                lic = row.get("license") or row.get("licenses")
                if lic:
                    license_counts[str(lic)[:60]] += 1
                if total >= max_bytes:
                    break
            if not docs:
                raise RuntimeError("stream produced 0 usable rows")
            return dict(name=name, docs=docs, licence_note=licence_note,
                        license_counts=dict(license_counts.most_common(10)),
                        n_docs_seen=n_rows)
        except Exception as e:
            tried.append((name, repr(e)))
            print(f"[corpus] {name} FAILED: {e!r} -- trying fallback" if len(tried) == 1
                  else f"[corpus] {name} ALSO FAILED: {e!r}")
    raise RuntimeError(f"no public code corpus reachable: {tried}")

_t0 = time.time()
CORPUS_INFO = load_code_corpus()
print(f"corpus loaded in {time.time()-_t0:.1f}s: {CORPUS_INFO['name']}  "
      f"docs={len(CORPUS_INFO['docs'])}  rows_scanned={CORPUS_INFO['n_docs_seen']}")
print("licence:", CORPUS_INFO["licence_note"])
if CORPUS_INFO["license_counts"]:
    print("per-file licence sample (top 10):", CORPUS_INFO["license_counts"])

class DocByteBatches:
    """Byte-level, split BY DOCUMENT (one row = one file), seeded --
    identical convention to tests/foreman/eval/r3_eval.py::DocByteBatches."""
    def __init__(self, docs, vocab_size=256, val_frac=0.1, split_seed=0):
        order = list(range(len(docs)))
        random.Random(split_seed).shuffle(order)
        cut = int(len(order) * (1 - val_frac))
        train_ix, val_ix = sorted(order[:cut]), sorted(order[cut:])
        def to_bytes(ix):
            t = "\n\n".join(docs[i] for i in ix)
            return torch.tensor(list(t.encode("utf-8", errors="ignore")), dtype=torch.long) % vocab_size
        self.train, self.val = to_bytes(train_ix), to_bytes(val_ix)
        self.n_docs, self.n_train_docs, self.n_val_docs = len(docs), len(train_ix), len(val_ix)

    def batch(self, split, bs, seq, gen, device):
        d = self.train if split == "train" else self.val
        i = torch.randint(len(d) - seq - 1, (bs,), generator=gen)
        x = torch.stack([d[j:j + seq] for j in i])
        y = torch.stack([d[j + 1:j + seq + 1] for j in i])
        return x.to(device), y.to(device)

CORPUS = DocByteBatches(CORPUS_INFO["docs"], VOCAB, val_frac=0.1, split_seed=SPLIT_SEED)
BYTES_TOTAL = len(CORPUS.train) + len(CORPUS.val)
CORPUS_TOKENS = BYTES_TOTAL   # byte-level: 1 byte == 1 token

def repetition_factor(n_params, tokens_per_param=20):
    return (tokens_per_param * n_params) / CORPUS_TOKENS

rep_a = repetition_factor(N_PARAMS_A)
rep_f = repetition_factor(N_PARAMS_F)
regime_a = "MEMORIZATION REGIME" if rep_a >= 1.0 else "generalization-admissible"
regime_f = "MEMORIZATION REGIME" if rep_f >= 1.0 else "generalization-admissible"
print(f"bytes_total={BYTES_TOTAL:,}  tokens={CORPUS_TOKENS:,}  "
      f"n_docs={CORPUS.n_docs} (train={CORPUS.n_train_docs} val={CORPUS.n_val_docs})  "
      f"BIGGER MODEL REPEATS MORE (n_params x20/tokens): "
      f"rep(a)={rep_a:.4f}x [{regime_a}]  rep(f)={rep_f:.4f}x [{regime_f}]")
log_row(label="corpus_pin", corpus=CORPUS_INFO["name"], bytes_total=BYTES_TOTAL,
        tokens=CORPUS_TOKENS, n_docs=CORPUS.n_docs, licence=CORPUS_INFO["licence_note"],
        rep_a=rep_a, rep_f=rep_f, regime_a=regime_a, regime_f=regime_f)

## Cell 5 -- training + eval, C27 hygiene

One `train_with_eval` shared by both arms: AdamW, grad-norm clip 1.0, a
FIXED held-out eval subsample (same eval generator seed for every
seed/arm), `del model; torch.cuda.empty_cache()` on return, and peak CUDA
memory logged so the next OOM is predicted rather than discovered.


In [ ]:
def eval_loss(model, corpus, *, batch, seq, eval_batches, split_seed, device):
    model.eval()
    tot = 0.0
    eg = torch.Generator().manual_seed(20260921 + split_seed)
    with torch.no_grad():
        for _ in range(eval_batches):
            xv, _ = corpus.batch("val", batch, seq, eg, device)
            _, loss = model(xv, labels=xv)
            tot += float(loss.detach())
    model.train()
    return tot / eval_batches

def train_with_eval(arm, *, steps, batch, seq, lr, seed, split_seed, device,
                     eval_batches=EVAL_BATCHES, log_every=0):
    torch.manual_seed(seed)
    model = build_arm(arm, hidden_size=HIDDEN, n_layers=LAYERS, n_heads=HEADS,
                       seq=seq, vocab_size=VOCAB).to(device)
    model.train()
    n_params = model.n_params()
    gen = torch.Generator().manual_seed(seed + 1)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    if device == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    losses = []
    t0 = time.time()
    for step in range(steps):
        x, _ = CORPUS.batch("train", batch, seq, gen, device)
        _, loss = model(x, labels=x)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        losses.append(float(loss.detach()))
        if log_every and (step % log_every == 0):
            print(f"  [{arm} seed={seed}] step {step}/{steps} loss={losses[-1]:.4f}")
    final_eval = eval_loss(model, CORPUS, batch=batch, seq=seq,
                            eval_batches=eval_batches, split_seed=split_seed, device=device)
    peak = torch.cuda.max_memory_allocated() if device == "cuda" else 0
    train_s = time.time() - t0
    del model
    if device == "cuda":
        torch.cuda.empty_cache()
    return dict(arm=arm, seed=seed, n_params=n_params, steps=steps,
                final_eval_loss=final_eval, loss_last_train=losses[-1],
                peak_bytes=int(peak), train_seconds=train_s)

## Cell 6 -- calibrate steps to the remaining budget

Chinchilla steps at the 20-tokens/param ceiling on arm (a)'s own count is
the pre-registered formula (`round(20*n_params_a/(batch*seq))`). A short
timed calibration run then checks whether the FULL 5-seed x 2-arm grid at
that step count fits the time actually left; if not, seeds are dropped
(never a run truncated mid-seed) and the drop is logged and printed, not
buried.


In [ ]:
N_SEEDS_PREREGISTERED = 5
CHINCHILLA_STEPS = max(1, round(20.0 * N_PARAMS_A / (BATCH * SEQ)))
print(f"Chinchilla steps (20x budget on arm-a param count): {CHINCHILLA_STEPS}")

CAL_STEPS = min(20, CHINCHILLA_STEPS)
_t0 = time.time()
_cal = train_with_eval("softmax", steps=CAL_STEPS, batch=BATCH, seq=SEQ, lr=LR,
                        seed=999, split_seed=SPLIT_SEED, device=DEVICE, eval_batches=2)
_cal_s = time.time() - _t0
sec_per_step = (_cal_s) / CAL_STEPS
print(f"calibration: {CAL_STEPS} steps in {_cal_s:.1f}s -> {sec_per_step:.4f} s/step "
      f"(includes one model build + eval; a slight overestimate per pair)")
log_row(label="calibration", cal_steps=CAL_STEPS, cal_seconds=_cal_s, sec_per_step=sec_per_step)

STEPS = CHINCHILLA_STEPS
remaining = budget_left()
# reserve the calibration pair's own already-spent time is implicit in remaining;
# estimate full-grid cost at N_SEEDS_PREREGISTERED and back off seeds first,
# then steps, to fit.
def grid_cost_s(n_seeds, steps):
    return 2 * n_seeds * steps * sec_per_step

N_SEEDS = N_SEEDS_PREREGISTERED
DROPPED = []
while N_SEEDS > 1 and grid_cost_s(N_SEEDS, STEPS) > remaining * 0.85:
    N_SEEDS -= 1
if grid_cost_s(N_SEEDS, STEPS) > remaining * 0.85:
    new_steps = max(200, int((remaining * 0.85) / (2 * N_SEEDS * sec_per_step)))
    if new_steps < STEPS:
        DROPPED.append(f"steps {STEPS} -> {new_steps} (Chinchilla ceiling not reached at N_SEEDS={N_SEEDS})")
        STEPS = new_steps
if N_SEEDS < N_SEEDS_PREREGISTERED:
    DROPPED.append(f"seeds {N_SEEDS_PREREGISTERED} -> {N_SEEDS} (5-seed grid did not fit the remaining budget)")

est_total = grid_cost_s(N_SEEDS, STEPS)
print(f"remaining budget: {remaining:.0f}s | planned grid: {N_SEEDS} seeds x 2 arms x {STEPS} steps "
      f"~= {est_total:.0f}s estimated")
if DROPPED:
    print("DROPPED to fit the 1-hour budget:", "; ".join(DROPPED))
else:
    print("full 5-seed pre-registered grid fits the budget -- nothing dropped")
log_row(label="grid_plan", n_seeds=N_SEEDS, steps=STEPS, estimated_seconds=est_total,
        remaining_budget_s=remaining, dropped=DROPPED)

## Cell 7 -- the driver: softmax vs hard-concrete, `N_SEEDS` seeds

INCREMENTAL: each (arm, seed) result is written to `kg_third_results.jsonl`
as soon as it finishes. `del model; torch.cuda.empty_cache()` already
happened inside `train_with_eval`. If the budget runs out mid-grid the
loop stops cleanly and says which pairs it did not reach, rather than
being cut off silently.


In [ ]:
landed = {"softmax": [], "hard_concrete": []}
pairs_run, pairs_skipped = [], []
_grid_t0 = time.time()

for seed in range(N_SEEDS):
    for arm_key, arm_name in (("softmax", "softmax"), ("smprime", "hard_concrete")):
        if budget_left() < 60:
            pairs_skipped.append((arm_name, seed))
            continue
        t1 = time.time()
        rec = train_with_eval(arm_key, steps=STEPS, batch=BATCH, seq=SEQ, lr=LR,
                               seed=seed, split_seed=SPLIT_SEED, device=DEVICE)
        dt = time.time() - t1
        row = log_row(label="seed_result", arm=arm_name, seed=seed,
                       n_params=rec["n_params"], steps=STEPS,
                       final_eval_loss=rec["final_eval_loss"],
                       loss_last_train=rec["loss_last_train"],
                       peak_mib=round(rec["peak_bytes"] / 1024**2, 2),
                       run_seconds=round(dt, 1))
        landed[arm_name].append(rec["final_eval_loss"])
        pairs_run.append((arm_name, seed))
        print(f"[{len(pairs_run)}/{2*N_SEEDS}] {arm_name} seed={seed} "
              f"final_eval_loss={rec['final_eval_loss']:.4f} peak={row['peak_mib']}MiB ({dt:.1f}s)")

print(f"\\npairs run: {len(pairs_run)}/{2*N_SEEDS}  |  skipped (budget): {pairs_skipped}")
log_row(label="driver_done", pairs_run=len(pairs_run), pairs_skipped=str(pairs_skipped),
        wall_clock_s=time.time() - _grid_t0)

## Cell 8 -- verdict: does the gated arm transfer to source code?

Reads `kg_third_results.jsonl` back (the same incremental file every cell
wrote to) and applies the pre-registered branch rule: TRANSFERS if
hard-concrete beats softmax by >0.01 nats at every completed seed pair;
DOES NOT TRANSFER on a tie or a softmax win; MIXED otherwise. No tuning
happened above this cell.


In [ ]:
n_pairs = min(len(landed["softmax"]), len(landed["hard_concrete"]))
diffs = [landed["hard_concrete"][i] - landed["softmax"][i] for i in range(n_pairs)]
TIE_BAND = 0.01
n_f_better = sum(1 for d in diffs if d < -TIE_BAND)
n_tie = sum(1 for d in diffs if abs(d) <= TIE_BAND)
n_a_better = sum(1 for d in diffs if d > TIE_BAND)

if n_pairs == 0:
    branch = "NO COMPLETED PAIRS -- budget exhausted before any seed finished both arms"
elif n_f_better == n_pairs:
    branch = "TRANSFERS -- hard-concrete beats softmax by >0.01 nats at every completed seed"
elif n_a_better == n_pairs:
    branch = "DOES NOT TRANSFER -- softmax twin wins at every completed seed on this corpus"
elif n_tie == n_pairs:
    branch = "DOES NOT TRANSFER -- tie within 0.01 nats at every completed seed"
else:
    branch = "MIXED -- effect is inside the seed spread on this corpus"

print("=" * 70)
print(f"CORPUS: {CORPUS_INFO['name']}  bytes_total={BYTES_TOTAL:,}  tokens={CORPUS_TOKENS:,}")
print(f"licence: {CORPUS_INFO['licence_note']}")
print(f"repetition_factor: a={rep_a:.4f}x [{regime_a}]  f={rep_f:.4f}x [{regime_f}]")
print(f"seeds completed (both arms): {n_pairs}/{N_SEEDS}  steps/run: {STEPS}")
print(f"accelerator (from torch): {ACCEL}")
print("-" * 70)
print("seed   softmax(a)   hard-cc(f)   diff(f-a)")
for i in range(n_pairs):
    print(f"{i:>4}   {landed['softmax'][i]:.4f}       {landed['hard_concrete'][i]:.4f}      {diffs[i]:+.4f}")
if diffs:
    mean_diff = sum(diffs) / len(diffs)
    print(f"mean   {sum(landed['softmax'][:n_pairs])/n_pairs:.4f}       "
          f"{sum(landed['hard_concrete'][:n_pairs])/n_pairs:.4f}      {mean_diff:+.4f}")
print("-" * 70)
print("BRANCH:", branch)
print("=" * 70)
if DROPPED:
    print("DROPPED to fit the 1-hour Kaggle budget:", "; ".join(DROPPED))
print("DECIDING NUMBERS COME FROM THE CERTIFIED LOCAL CARD. "
      "This cell reproduces on a third, structurally distinct domain; "
      "it does not replace a local reading.")

log_row(label="verdict", n_pairs=n_pairs, n_seeds_planned=N_SEEDS,
        diffs=diffs, mean_diff=(sum(diffs)/len(diffs) if diffs else None),
        n_f_better=n_f_better, n_tie=n_tie, n_a_better=n_a_better,
        branch=branch, corpus=CORPUS_INFO["name"], bytes_total=BYTES_TOTAL,
        tokens=CORPUS_TOKENS, rep_a=rep_a, rep_f=rep_f, dropped=DROPPED)